In [0]:
import json 
import os 
from urllib.parse import urljoin

import pandas as pd 
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/' 
FILES_URL = 'https://cvijan.github.io/nss-api/files/' 
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True) 
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes unchanged
    return len(response.content)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# get_json(BASE_URL + 'orders/page-99.json')
get_json(BASE_URL + 'status.json')['service']


'Northern Star Shop API'

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    
    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(fetch_all(BASE_URL + 'orders/page-1.json'))

1852

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path: str) -> None: 
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције
# download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

# За разлику од get_json, ова функција не чита садржај, већ само копира фајл на диск, бајт по
# бајт
# Читање долази после, са pd.read_csv и pd.read_parquet
# Parquet није обичан текст, па се уписује са 'wb', а не са 'w'
# Проба: download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,
# 'teams.csv'))
# Повратна вредност: 801, а у data/raw се појављује фајл teams.csv

download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR, 'teams.csv'))

801

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције
# download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR, 'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR, 'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR, 'matches.parquet'))

    orders = pd.json_normalize(orders, sep='_').rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    products = pd.read_csv(FILES_URL + 'products.csv')
    teams = pd.read_csv(FILES_URL + 'teams.csv')
    matches = pd.read_parquet(FILES_URL + 'matches.parquet')

    return {
        'orders': orders,
        'products': products,
        'teams': teams,
        'matches': matches
    }

data = extract(RAW_DIR)

orders = data['orders']



In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред
# остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало
# постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

products = data['products']

def clean_products(df: pd.DataFrame) -> pd.DataFrame: 
    clean = df.copy()
    clean = clean[~clean.duplicated(subset='product_id', keep='first')]

    clean['team_code'] = clean['team_code'].str.strip().str.upper()
    clean['category'] = clean['category'].str.strip().str.lower()

    clean['price'] = clean['price'].astype(str).str.replace('RSD', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False).astype(float)

    clean['active'] = clean['active'].str.strip().str.lower().eq('yes')

    return clean


,product_id,sku,product_name,team_code,category,price,active
0,1001,NPM-JER-1001,NPM Home Jersey,NPM,jerseys,7490.0,True
1,1002,NPM-JER-1002,NPM Away Jersey,NPM,jerseys,6990.0,True
2,1003,NPM-SCA-1003,NPM Scarf,NPM,scarves,1490.0,True
3,1004,NPM-CAP-1004,NPM Cap,NPM,caps,2290.0,True
4,1005,NPM-ACC-1005,NPM Mug,NPM,accessories,990.0,True
...,...,...,...,...,...,...,...
100,1101,NSL-BAL-1101,NSL Match Ball,NaN,balls,7490.0,True
101,1102,NSL-BAL-1102,NSL Training Ball,NaN,balls,2990.0,True
102,1103,NSL-ACC-1103,NSL Water Bottle,NaN,accessories,890.0,True
103,1104,NSL-ACC-1104,NSL Backpack,NaN,accessories,3990.0,True


In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()

    for col in ['order_id', 'customer', 'city', 'payment']:
        clean[col] = clean[col].astype(str).str.strip()
    
    clean['order_id'] = clean['order_id'].astype(str).str.upper()

    for col in ['customer', 'city']:
        clean[col] = clean[col].astype(str).str.title()
        
    clean.loc[clean['city'].isna(), 'city'] = 'Uknown'

    return clean

clean_orders(orders)


,order_id,order_date,product_id,quantity,unit_price,payment,customer,city
0,NSS-00001,2026-08-15,1002,1,6990.0,cash,Nikola Stojanović,Zrenjanin
1,NSS-00002,2026-08-15,1094,3,1290.0,card,Jovana Pavlović,Čačak
2,NSS-00003,2026-08-15,1041,1,6990.0,card,Sara Živković,Beograd
3,NSS-00004,2026-08-15,1075,1,990.0,cash,Sara Đorđević,Subotica
4,NSS-00005,2026-08-15,1044,1,1290.0,card,Stefan Popović,Niš
...,...,...,...,...,...,...,...,...
1847,NSS-01836,2026-11-15,1046,3,7490.0,card,Milica Popović,Subotica
1848,NSS-01837,2026-11-15,1024,1,1290.0,card,Marko Kovačević,Čačak
1849,NSS-01838,2026-11-15,1042,2,6490.0,cash,Marko Nikolić,Čačak
1850,NSS-01839,2026-11-15,1004,2,2290.0,cash,Jelena Živković,Zrenjanin


In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    df = df.copy()
    duplicates = df[df.duplicated(subset='order_id', keep='first')]
    uniques = df.drop_duplicates(subset='order_id', keep='first')

    return uniques, duplicates

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа
# 'OK' за исправан ред, у супротном разлог
# Провере радити редом и вратити први пронађени проблем
# order_id не почиње са NSS- -> 'invalid_order'
# quantity недостаје -> 'missing_quantity'
# quantity није цео број већи од нуле -> 'invalid_quantity'
# unit_price није број већи од нуле -> 'invalid_price'
# product_id не постоји међу производима -> 'unknown_product'
# Помоћ:
# pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
# pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    unit_price = pd.to_numeric(row['unit_price'], errors='coerce')

    if not row['order_id'].startswith('NSS'):
        return 'invalid_order'
    elif pd.isna(quantity):
        return 'missing_quantity'
    elif pd.isna(quantity) or quantity <= 0 or quantity % 1 != 0:
        return 'invalid_quantity'
    elif pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'
    elif not row['product_id'] in product_ids:
        return 'uknown_product'
    
    return 'OK'

In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# pd.DataFrame
# Типови:
# quantity -> int
# unit_price -> float
# order_date -> датум (pd.to_datetime)
# Нова колона total (quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category
# (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = df['quantity'].astype(int)
    df['unit_price'] = df['unit_price'].astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']

    cleaned_products = clean_products(products)

    # print(cleaned_products)

    df = pd.merge(
        df,
        cleaned_products[['product_id', 'product_name', 'team_code', 'category']],
        on='product_id',
        how='left'
    )

    return df

In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,
# rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason
# (дупликати за разлог треба да имају 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

# orders = orders.to_dict(orient='records')
# print(orders)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = df.copy()
    products = products.copy()

    cleaned_products = clean_products(products)
    df = clean_orders(df)

    product_ids = set(cleaned_products['product_id'].unique())

    valid_orders = []
    invalid_orders = []

    valid, rejected = remove_duplicate_orders(df)
    for order in valid.to_dict(orient='records'):
        result = validate_order(order, product_ids)

        if result == 'OK':
            valid_orders.append(order)
        else:
            invalid_orders.append({
                **order,
                'reason': result
            })

    invalid_orders.append(rejected)
    valid_df = pd.DataFrame(valid_orders)
    valid_df = transform_orders(valid_df, products)

    
    return valid_df, pd.DataFrame(invalid_orders)

valid, invalid = split_orders(orders, products)

invalid

,order_id,order_date,product_id,quantity,unit_price,payment,customer,city,reason
0,NSS-00034,2026-08-16,1031,None,7490.0,card,Ivana Petrović,Kraljevo,missing_quantity
1,TST-00077,2026-08-18,1098,1,1490.0,card,Petar Đorđević,Sombor,invalid_order
2,TST-00084,2026-08-19,1078,1,1990.0,card,Đorđe Jovanović,Kragujevac,invalid_order
3,NSS-00108,2026-08-20,1085,0,1490.0,paypal,Milan Pavlović,Čačak,invalid_quantity
4,NSS-00131,2026-08-21,1022,None,6490.0,card,Nikola Đorđević,Čačak,missing_quantity
5,TST-00164,2026-08-23,1073,1,1490.0,card,Katarina Kovačević,Beograd,invalid_order
6,NSS-00213,2026-08-25,1013,None,1490.0,card,Ivana Ilić,Novi Sad,missing_quantity
7,NSS-00289,2026-08-29,1019,1,0.0,card,Sara Đorđević,Kragujevac,invalid_price
8,NSS-00322,2026-08-31,1014,1,None,cash,Milica Ilić,Subotica,invalid_price
9,NSS-00398,2026-09-04,1070,0,1490.0,cash,Sara Kovačević,Niš,invalid_quantity


In [0]:
# 12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams:
# pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, један
# за домаћина и један за госта.
# Користити само утакмице са status == 'finished' (одложена нема резултат)
# Колоне:
# match_id
# match_date (Датум из kickoff_ts, без времена -
# df['kickoff_ts'].dt.normalize())
# team_code, opponent_code
# goals_for, goals_against
# result ('W', 'D', 'L'), points (3, 1, 0)
# Утакмице имају team_id, а производи team_code

matches = data['matches']
teams = data['teams']


def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:

    df = matches[matches['status'] == 'finished'].copy()
    df['match_date'] = df['kickoff_ts'].dt.normalize()

    tm = teams.copy()

    home_teams = tm.rename(columns={'team_id': 'home_team_id', 'code': 'team_code'})
    away_teams = tm.rename(columns={'team_id': 'away_team_id', 'code': 'oponent_code'})

    merged = pd.merge(
        df,
        home_teams,
        on='home_team_id',
        how='left'
    )

    merged = merged.merge(
        away_teams,
        on='away_team_id',
        how='left'
    )

    # print(merged.columns)
    home = merged[['match_id', 'match_date', 'team_code', 'oponent_code', 'home_goals', 'away_goals']]
    home = home.rename(columns={'home_goals': 'goals_for', 'away_goals': 'goals_against'})

    away = merged[['match_id', 'match_date', 'team_code', 'oponent_code', 'home_goals', 'away_goals']]
    away = away.rename(columns={'home_goals': 'goals_against', 'away_goals': 'goals_for'})

    m = pd.concat([home, away], ignore_index=True)

    # print(m)
    wins = (
        (m['goals_for'] > m['goals_against'])
        |
        (m['goals_for'] < m['goals_against'])
    )

    draws = (
        m['goals_for'] == m['goals_against']
    )

    m.loc[wins, 'result'] = 'W'
    m.loc[draws, 'result'] = 'D'
    m.loc[~(wins | draws), 'result'] = 'L'

    m.loc[m['result'] == 'W', 'points'] = 3
    m.loc[m['result'] == 'D', 'points'] = 1
    m.loc[m['result'] == 'L', 'points'] = 0

    return m

team_results(matches, teams)


,match_id,match_date,team_code,oponent_code,goals_for,goals_against,result,points
0,2627001,2026-08-15,BRU,FWR,2,1,W,3.0
1,2627002,2026-08-15,CHS,VDC,0,1,W,3.0
2,2627003,2026-08-15,MBA,WFT,3,0,W,3.0
3,2627004,2026-08-15,NPM,IVU,1,0,W,3.0
4,2627005,2026-08-15,PCU,HGA,1,1,D,1.0
...,...,...,...,...,...,...,...,...
233,2627086,2026-10-24,KLT,NPM,3,1,W,3.0
234,2627087,2026-10-24,WFT,RVA,2,0,W,3.0
235,2627088,2026-10-25,PCU,CHS,1,3,W,3.0
236,2627089,2026-10-25,FWR,IVU,0,2,W,3.0


In [0]:
# 13. Табела лиге. Написати функцију league_table(results: pd.DataFrame) -> pd.DataFrame
# која од резултата из задатка 12 прави табелу
# Колоне:
# position, team_code
# played, won, drawn, lost
# goals_for, goals_against, goal_diff
# points
# Редослед табеле: Бодови, па гол разлика, па дати голови (све опадајуће)
# Output: 20 тимова, први FWR (26 бодова), други BMC (23)
# HGA и OKT имају по 11 утакмица

results = team_results(matches, teams)
print(results.columns)

def league_table(results: pd.DataFrame) -> pd.DataFrame:
    print(results)

league_table(results)

Index(['match_id', 'match_date', 'team_code', 'oponent_code', 'goals_for',
       'goals_against', 'result', 'points'],
      dtype='object')
     match_id match_date team_code  ... goals_against  result  points
0     2627001 2026-08-15       BRU  ...             1       W     3.0
1     2627002 2026-08-15       CHS  ...             1       W     3.0
2     2627003 2026-08-15       MBA  ...             0       W     3.0
3     2627004 2026-08-15       NPM  ...             0       W     3.0
4     2627005 2026-08-15       PCU  ...             1       D     1.0
..        ...        ...       ...  ...           ...     ...     ...
233   2627086 2026-10-24       KLT  ...             1       W     3.0
234   2627087 2026-10-24       WFT  ...             0       W     3.0
235   2627088 2026-10-25       PCU  ...             3       W     3.0
236   2627089 2026-10-25       FWR  ...             2       W     3.0
237   2627090 2026-10-25       MBA  ...             2       D     1.0

[238 rows x 8 col